# NLU Shared Task — Demo Solution 1 (Category A)

This notebook generates predictions for the AV test set using the trained Solution 1 stacking ensemble.

**Pipeline:** Raw test CSV → spaCy preprocessing → stylometric features (84) → info-theoretic features (8) → General Impostor features (5) → stacking ensemble → predictions

In [1]:
# Install required packages
!pip install spacy lightgbm textstat joblib tqdm scikit-learn pandas numpy scipy gdown
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 7.8 MB/s  0:00:01 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure src is in the Python path
sys.path.insert(0, os.path.abspath('..'))

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Setup: Download Required Model Artefacts
Before running inference, download the two large artefacts from Google Drive.
This only needs to run once.

In [3]:
import gdown

models_dir = Path('../models/solution1')
features_dir = Path('../data/features')
models_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)

# Download model_a2.joblib (LightGBM base model)
model_a2_path = models_dir / 'model_a2.joblib'
if not model_a2_path.exists():
    print('Downloading model_a2.joblib ...')
    gdown.download(
        url='https://drive.google.com/file/d/1rWwn__rn9BwFgdaxINDbklYOCgHquY1z/view?usp=sharing',
        output=str(model_a2_path),
        fuzzy=True,
    )
    print('Downloaded model_a2.joblib')
else:
    print('model_a2.joblib already present, skipping.')

# Download gi_corpus_vectors.joblib (GI impostor corpus)
corpus_path = features_dir / 'gi_corpus_vectors.joblib'
if not corpus_path.exists():
    print('Downloading gi_corpus_vectors.joblib ...')
    gdown.download(
        url='https://drive.google.com/file/d/193TCG1-I4zSeW2t0peWCi3hxAPZ2FIeX/view?usp=sharing',
        output=str(corpus_path),
        fuzzy=True,
    )
    print('Downloaded gi_corpus_vectors.joblib')
else:
    print('gi_corpus_vectors.joblib already present, skipping.')

Downloading...
From: https://drive.google.com/uc?id=1rWwn__rn9BwFgdaxINDbklYOCgHquY1z
To: /Users/alexiosphilalithis/Documents/Uni mac yr3/NLU/NLU_CW/models/solution1/model_a2.joblib
100%|██████████| 46.0M/46.0M [00:06<00:00, 6.84MB/s]


Downloaded model_a2.joblib


Downloading...
From (original): https://drive.google.com/uc?id=193TCG1-I4zSeW2t0peWCi3hxAPZ2FIeX
From (redirected): https://drive.google.com/uc?id=193TCG1-I4zSeW2t0peWCi3hxAPZ2FIeX&confirm=t&uuid=f3933799-805e-496d-b413-7736507fc56b
To: /Users/alexiosphilalithis/Documents/Uni mac yr3/NLU/NLU_CW/data/features/gi_corpus_vectors.joblib
100%|██████████| 227M/227M [00:48<00:00, 4.66MB/s] 


Downloaded gi_corpus_vectors.joblib


## 1. Configure Paths

In [5]:
TEST_CSV     = Path('../data/test_data/AV/test.csv')
MODEL_DIR    = Path('../models/solution1')
FEATURES_DIR = Path('../data/features')
OUTPUT_PATH  = Path('../outputs/Group_17_A.csv')
N_JOBS       = 4

# Sanity checks
assert TEST_CSV.exists(),                                    f'Test CSV not found: {TEST_CSV}'
assert (MODEL_DIR / 'model_a2.joblib').exists(),             f'model_a2.joblib not found in {MODEL_DIR}'
assert (FEATURES_DIR / 'gi_corpus_vectors.joblib').exists(), f'gi_corpus_vectors.joblib not found in {FEATURES_DIR}'

print('All required files found.')
print(f'Test CSV: {pd.read_csv(TEST_CSV).shape[0]} pairs')

All required files found.
Test CSV: 5985 pairs


## 2. Generate Predictions

Runs the full pipeline end-to-end: preprocessing → feature extraction → stacking ensemble.

The GI features use the saved training corpus as the impostor pool (consistent with training).

In [7]:
from src.solution1.predict import predict_from_csv

submission_df = predict_from_csv(
    test_csv=TEST_CSV,
    model_dir=MODEL_DIR,
    features_dir=FEATURES_DIR,
    n_jobs=N_JOBS,
)

print(f'Generated {len(submission_df)} predictions.')
print(f'Class distribution: {submission_df["prediction"].value_counts().to_dict()}')

Preprocessing with spaCy ...


spaCy:   0%|          | 0/5985 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently ins

Extracting stylometric features (84) ...


stylometric:   2%|▏         | 104/5985 [00:01<01:20, 73.18it/s]/Users/alexiosphilalithis/Documents/Uni mac yr3/NLU/NLU_CW/src/solution1/training/feature_extraction.py:61: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_a, p_b)
/Users/alexiosphilalithis/Documents/Uni mac yr3/NLU/NLU_CW/src/solution1/training/feature_extraction.py:61: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_a, p_b)
/Users/alexiosphilalithis/Documents/Uni mac yr3/NLU/NLU_CW/src/solution1/training/feature_extraction.py:61: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_a, p_b)
/Users/alexiosphilalithis/Documents/Uni mac yr3/NLU/NLU_CW/src/solution1/training/feature_extraction.py:61: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_a, p_b)
stylo

Extracting info-theoretic features (8) ...


info-theoretic: 100%|██████████| 5985/5985 [00:01<00:00, 3458.40it/s]
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.3.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.3.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loading GI corpus ...
  corpus shape: (55286, 5000)
Computing GI features (5) ...


GI: 100%|██████████| 5985/5985 [05:10<00:00, 19.30it/s]
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Feature matrix: (5985, 97)
Generated 5985 predictions.
Class distribution: {1: 4235, 0: 1750}


## 3. Save Output

Saves predictions to a CSV with a single `prediction` column (0 or 1) as required by the submission spec.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to {OUTPUT_PATH}')
display(submission_df.head(10))

Saved to ../outputs/Group_17_A.csv


,prediction
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,0
8,1
9,1
